# 09 — Grad-CAM and Interpretability

Where does the ResNet50 look when it calls a vehicle *damaged*? Grad-CAM weights the final convolutional feature maps by the gradient of the damage logit, giving a heatmap over the image. This is a diagnostic, not a proof: it shows the regions that drove the prediction. If the model keys on the damaged panel we gain confidence; if it keys on background, sky, or watermarks, that is a dataset-bias warning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("c:\\Users\\Niraj Mhatre\\projects\\motor_insurance_data\\Fast_Furious_Insured")
TRAIN_IMG_DIR = DATA_DIR / "trainImages"
TRAIN_METADATA = DATA_DIR / "processed" / "train_metadata_clean.csv"
MODEL_DIR = Path("../models")

import pandas as pd
df = pd.read_csv(TRAIN_METADATA)

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(torch.load(MODEL_DIR / "resnet50_best.pth", map_location=device))
model = model.to(device)
model.eval()

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Grad-CAM via forward / backward hooks on `layer4`

`layer4` is the last residual block — its 7×7 feature maps retain enough spatial structure to localise damage while carrying high-level semantics.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, input_tensor):
        self.model.zero_grad()
        logit = self.model(input_tensor)
        logit.backward(torch.ones_like(logit))

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)

        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        prob = torch.sigmoid(logit).item()
        return cam, prob


grad_cam = GradCAM(model, model.layer4[-1])

In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def cam_for_image(image_name):
    image = Image.open(TRAIN_IMG_DIR / image_name).convert("RGB")
    tensor = val_transform(image).unsqueeze(0).to(device)
    cam, prob = grad_cam(tensor)
    base = denormalize(tensor.squeeze(0).cpu()).permute(1, 2, 0).numpy()
    return base, cam, prob

In [ ]:
def show_cam_grid(image_names, titles=None):
    n = len(image_names)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    axes = np.atleast_2d(axes)
    for j, name in enumerate(image_names):
        base, cam, prob = cam_for_image(name)
        axes[0, j].imshow(base)
        axes[0, j].axis("off")
        head = titles[j] if titles else ""
        axes[0, j].set_title(f"{head}\nP(damaged)={prob:.2f}")

        axes[1, j].imshow(base)
        axes[1, j].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, j].axis("off")
        axes[1, j].set_title("Grad-CAM")
    plt.tight_layout()
    plt.show()

## Damaged vehicles

In [ ]:
damaged = df[df["Condition"] == 1].sample(4, random_state=42)
show_cam_grid(damaged["Image_path"].tolist(), ["damaged"] * len(damaged))

## Undamaged vehicles

In [ ]:
undamaged = df[df["Condition"] == 0].sample(4, random_state=7)
show_cam_grid(undamaged["Image_path"].tolist(), ["undamaged"] * len(undamaged))

## Disagreement cases

The heatmaps are most informative where the model is wrong or unsure. We pull a few records where the predicted probability contradicts the recorded condition and inspect what the network attended to.

In [ ]:
subset = df.sample(min(200, len(df)), random_state=1).reset_index(drop=True)

probs = []
for name in subset["Image_path"]:
    tensor = val_transform(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        probs.append(torch.sigmoid(model(tensor)).item())
subset["prob"] = probs
subset["pred"] = (subset["prob"] >= 0.5).astype(int)

mistakes = subset[subset["pred"] != subset["Condition"]]
mistakes = mistakes.reindex(mistakes["prob"].sub(0.5).abs().sort_values(ascending=False).index)

if len(mistakes) > 0:
    picks = mistakes.head(4)
    show_cam_grid(picks["Image_path"].tolist(),
                  [f"actual {int(c)}" for c in picks["Condition"]])
else:
    print("No misclassifications in the sampled subset.")

## Interpretation

Grad-CAM provides a visual diagnostic of the regions that contributed to each prediction — nothing stronger. Concentrated activation on dented or scratched panels supports the claim that the classifier learned damage-relevant features; diffuse activation on background or borders would flag a spurious shortcut and motivate re-examining the data collection. It does not prove the network "understands" damage in any causal sense.